In [8]:
import os
import pandas as pd

# Get the directory where the notebook is (the /notebooks folder)
notebook_dir = os.getcwd()

# Go up one level to /lanl-project
project_root = os.path.dirname(notebook_dir)

# Now point to /data
DATA_DIR = os.path.join(project_root, "data")
REDTEAM_PATH = os.path.join(DATA_DIR, "redteam.txt.gz")
AUTH_PATH = os.path.join(DATA_DIR, "auth.txt.gz")

print(f"Project Root: {project_root}")
print(f"Looking for Redteam at: {REDTEAM_PATH}")

if os.path.exists(REDTEAM_PATH):
    print("✅ Found it! Loading redteam data...")
    redteam_df = pd.read_csv(REDTEAM_PATH, names=['time', 'user', 'src_comp', 'dst_comp'])
    print(f"Loaded {len(redteam_df)} redteam events.")
else:
    print("❌ Still not found. Run '%ls ..' in a cell to see what's in the parent folder.")

Project Root: /Users/williammumper/lanl-project
Looking for Redteam at: /Users/williammumper/lanl-project/data/redteam.txt.gz
✅ Found it! Loading redteam data...
Loaded 749 redteam events.


In [9]:
import tqdm # For a progress bar

# 1. Get the list of unique compromised users
red_user_set = set(redteam_df['user'].unique())

# 2. Prepare the extraction
auth_cols = ['time', 'src_user', 'dst_user', 'src_comp', 'dst_comp', 'auth_type', 'logon_type', 'orient', 'success']
adversarial_activity = []

print(f"Starting scan for {len(red_user_set)} users...")

# 3. Stream through auth.txt.gz in chunks
# We use chunksize to keep RAM usage low (8GB M2 limit)
chunks = pd.read_csv(AUTH_PATH, names=auth_cols, chunksize=1000000, compression='gzip')

for i, chunk in enumerate(chunks):
    # Filter: Keep rows where either the source or destination is a redteam user
    mask = chunk['src_user'].isin(red_user_set) | chunk['dst_user'].isin(red_user_set)
    match = chunk[mask].copy()
    
    if not match.empty:
        adversarial_activity.append(match)
        
    if i % 10 == 0:
        print(f"Proceassed {i} million rows... Found {sum(len(x) for x in adversarial_activity)} matches.")

# 4. Consolidate and Save
adv_df = pd.concat(adversarial_activity).sort_values('time')
context_output_path = os.path.join(DATA_DIR, "redteam_context.csv")
adv_df.to_csv(context_output_path, index=False)

print(f"\n✅ DONE! Extracted {len(adv_df)} events.")
print(f"Saved to: {context_output_path}")

Starting scan for 104 users...
Processed 0 million rows... Found 32457 matches.
Processed 10 million rows... Found 238048 matches.
Processed 20 million rows... Found 439401 matches.
Processed 30 million rows... Found 621623 matches.
Processed 40 million rows... Found 821849 matches.
Processed 50 million rows... Found 1018409 matches.
Processed 60 million rows... Found 1231698 matches.
Processed 70 million rows... Found 1430468 matches.
Processed 80 million rows... Found 1621597 matches.
Processed 90 million rows... Found 1800907 matches.
Processed 100 million rows... Found 1989166 matches.
Processed 110 million rows... Found 2162250 matches.
Processed 120 million rows... Found 2345032 matches.
Processed 130 million rows... Found 2518366 matches.
Processed 140 million rows... Found 2693795 matches.
Processed 150 million rows... Found 2873286 matches.
Processed 160 million rows... Found 3079890 matches.
Processed 170 million rows... Found 3258441 matches.
Processed 180 million rows... Fo